# Phase 3: Preprocessing & Augmentation Pipeline

## 1. Objective

This notebook builds the preprocessing pipeline that transforms verified raw data (Phases 1-2) into a form ready for model training — a leakage-free train/validation split, a PyTorch `Dataset` class, and an augmentation strategy — before any modeling begins in Phase 4.

**Starting point:** Phase 2 finalized the training set as `hurricane-harvey` + `hurricane-michael` + `santa-rosa-wildfire`, with `mexico-earthquake` reserved for Phase 7. This phase works exclusively with the 3 confirmed training disasters.

## 2. What This Notebook Covers

1. **3.1** — Train/validation split by location ID (not by file), keeping every location's full file bundle (images, labels, targets, pre/post) together
2. **3.2** — Decision on handling `un-classified` labels for classifier training
3. **3.3** — PyTorch `Dataset` class for lazy loading of image/label pairs
4. **3.4** — Input size decision given hardware constraints
5. **3.5** — Augmentation pipeline design
6. **3.6** — Handling building density variation (from Phase 2.7's finding)
7. **3.7** — Visual verification of a processed batch

## 3.1 Train/Validation Split

**Task:** Split the combined training set (`hurricane-harvey` + `hurricane-michael` + `santa-rosa-wildfire`) into training and validation subsets, split by unique location ID rather than by individual file, so that every file belonging to one location (pre/post images, pre/post labels, pre/post targets) stays together in the same subset.

**Why by location ID, not by file:** each location's 6 related files (images, labels, targets — pre and post) must be treated as one indivisible unit. Splitting file types independently risks a location's pre-image landing in training while its post-image lands in validation, breaking the pairing the model depends on.

**Why only `labels/` was scanned to collect IDs:** Phase 1 already confirmed full parity (equal counts, no mismatches) across `images/`, `labels/`, and `targets/` for all disasters. Since every location's post-disaster label file existence guarantees its matching image and target files also exist, scanning `labels/` alone is sufficient — scanning all three folders would be redundant, since it would only reconfirm a fact already established in Phase 1.

In [1]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from PIL import Image
import numpy as np
import sys
sys.path.append(str(Path("..").resolve()))

DISASTERS = [
    "hurricane-harvey",
    "hurricane-michael",
    "santa-rosa-wildfire"
]

LABELS_DIR = Path("../data/raw/train/labels")

all_location_ids = []
for disaster in DISASTERS:
    pattern = f"*{disaster}*post_disaster.json"
    json_files = LABELS_DIR.glob(pattern)
    
    for file_path in json_files:
        # e.g. "hurricane-harvey_00000042_post_disaster.json" -> "hurricane-harvey_00000042"
        location_id = file_path.stem.replace("_post_disaster", "") # .stem gives filename without extension
        all_location_ids.append(location_id)

train_ids, val_ids = train_test_split(all_location_ids, test_size=0.2, random_state=42)        
print(f"Train: {len(train_ids)} locations, Validation: {len(val_ids)} locations")

Train: 710 locations, Validation: 178 locations


### Observations

`all_location_ids` was correctly split into an 80/20 train/validation ratio: **710** training locations and **178** validation locations, from a combined pool across all 3 training disasters.

Only the `labels/` folder was scanned to collect location IDs, since Phase 1 already confirmed that `images/`, `labels/`, and `targets/` share identical IDs (differing only in filename suffix and extension) with full parity across all three. This means any image or target file needed later can be reconstructed directly from an ID using the known naming pattern, rather than needing to load or store full file objects now — keeping this step lightweight (string IDs only) rather than memory-intensive.

## 3.2 Handling `Un-Classified` Labels

**Task:** Decide how to handle the `un-classified` damage label — not a genuine damage severity level, but an annotator's uncertainty marker (Phase 2.5-2.6 found these tend to be smaller, more visually ambiguous buildings) — before training the damage classifier.

**Decision:** Exclude `un-classified` buildings entirely from classifier training data. This project's scope is a 4-class damage severity classifier (No Damage / Minor / Major / Destroyed); `un-classified` doesn't represent a damage level at all, so training on it would ask the model to learn a category outside its intended prediction space. Adding it as a genuine 5th "uncertain" class was considered and rejected — that's a fundamentally different problem (out-of-distribution/uncertainty detection) and would expand this project's scope beyond its defined goal.

In [2]:
from collections import defaultdict
import json 
from shapely.wkt import loads as wkt_loads

def get_area_per_damage_class(disaster_name, labels_dir):
    """Returns the areas for each damage class."""
    pattern = f"*{disaster_name}*post_disaster.json" 
    json_files = labels_dir.glob(pattern)
    area_by_class = defaultdict(list)
    
    for file_path in json_files:
        with open(file_path, "r") as f:
            d = json.load(f)
            for building in d.get("features", {}).get("xy", []):
                wkt_str = building["wkt"]
                subtype = building.get("properties", {}).get("subtype") # damage_class
                if not subtype:
                    subtype = "un-classified"
                polygon = wkt_loads(wkt_str)
                area_by_class[subtype].append(polygon.area)
                
    return dict(area_by_class)

combined_area_by_class = defaultdict(list)                
for disaster in DISASTERS:
    result = get_area_per_damage_class(disaster, LABELS_DIR)
    for damage_class, areas in result.items():
        combined_area_by_class[damage_class].extend(areas)

for damage_class, areas in combined_area_by_class.items():
    print(f"{damage_class}: {len(areas)} buildings")

# Excluding logic
total_combined = sum(len(v) for v in combined_area_by_class.values())
excluded_count = len(combined_area_by_class["un-classified"])  
excluded_pct = excluded_count / total_combined * 100
print(f"Excluding {excluded_count} un-classified buildings ({excluded_pct:.2f}% of {total_combined} total)")


major-damage: 10203 buildings
minor-damage: 7948 buildings
no-damage: 35296 buildings
un-classified: 574 buildings
destroyed: 4629 buildings
Excluding 574 un-classified buildings (0.98% of 58650 total)


### Observations

Across the combined training set (3 disasters), `un-classified` accounts for only 574 buildings out of 58,650 total (0.98%) — a negligible fraction compared to the 4 real damage classes, which range from 4,629 (`destroyed`) to 35,296 (`no-damage`).

This confirms the exclusion decision costs almost nothing in terms of lost training signal. Even independent of the sample size, `un-classified` was never a genuine damage category the model should predict — it's an artifact of annotator uncertainty rather than a real class in this project's 4-class problem definition (No Damage / Minor / Major / Destroyed). The small percentage simply reinforces that excluding it is low-cost as well as conceptually correct.

**Decision finalized:** `un-classified` buildings are excluded from all classifier training, validation, and evaluation going forward.

## 3.3 PyTorch `SegmentationDataset` Class

**Task:** Build a PyTorch-style `Dataset` class that supplies one training example at a time (a pre-disaster image and its matching building-location mask) for the segmentation model, following PyTorch's standard `__len__` / `__getitem__` contract for lazy, on-demand data loading.

**Design decisions:**
- **Separate classes for segmentation vs. classification.** `SegmentationDataset` (this section) only needs "where are the buildings," not damage information — classification will use its own `ClassificationDataset` in Phase 5, since the two tasks need structurally different data.
- **Pre-disaster images only.** Buildings are intact and cleanly shaped pre-disaster, giving the segmentation model the clearest possible signal for what a building's outline looks like — post-disaster imagery (collapsed structures, smoke, debris) would introduce unnecessary noise for a task that doesn't need damage information at all. This matches the standard xView2 challenge approach of training localization separately from damage classification.
- **`un-classified` exclusion (from Section 3.2) does not apply here.** That decision concerns damage labels, which segmentation targets don't contain — the target masks are purely binary (building present or not), confirmed in this section's verification. The exclusion will be implemented in Phase 5's `ClassificationDataset` instead.

**Placement:** the class lives in `src/segmentation_dataset.py` (reusable pipeline code), not in the notebook itself, matching the project's established `src/` vs. `notebooks/` split. Paths are built relative to the file's own location (`Path(__file__).resolve().parent.parent`) rather than the caller's working directory, so the class remains correctly importable from any future location (e.g. Phase 10's FastAPI app).

In [9]:
from src.segmentation_dataset import SegmentationDataset
    
train_dataset = SegmentationDataset(train_ids)
val_dataset = SegmentationDataset(val_ids)

image, target = train_dataset[0] # can be checked for val_ids as well
print("Image size:", image.size, "mode:", image.mode)
print("Target size:", target.size, "mode:", target.mode)

image.show()
target_array = np.array(target)
print("Target unique pixel values:", np.unique(target_array))  

Image size: (1024, 1024) mode: RGB
Target size: (1024, 1024) mode: L
Target unique pixel values: [0 1]


### Observations

`SegmentationDataset`, applied to both `train_ids` (710 locations) and `val_ids` (178 locations), correctly returns paired examples via indexing (`dataset[0]`). Verification of the first training example confirmed:
- **Image:** 1024×1024, RGB (3-channel color) — a genuine pre-disaster satellite photo showing intact buildings, roads, and vegetation
- **Target mask:** 1024×1024, single-channel grayscale (`"L"` mode), with only two unique pixel values (`0`, `1`) — confirming the mask is a pure binary building-location map, containing no damage-class information

This confirms the target masks are structurally independent of damage labels, validating the decision to build separate `Dataset` classes for segmentation and classification rather than one combined class. `SegmentationDataset` is complete and ready to be used with a PyTorch `DataLoader` in Phase 4, where it will supply batched (image, target) pairs to train the building-localization model.